# SoundScape AI — Connect trained Computer Vision to the live website

**Purpose:** Export your *already trained* MobileNetV3 checkpoint (`scene_mobilenetv3.pt`) into two files readable by the Render **CPU** web app. **No retraining is required.**

The CV model classifies six scenes: buildings, forest, glacier, mountain, sea, street. It does **not** detect rain/wind or infer actual recorded sound. Your existing platform maps the output to suggested audio settings, which users can adjust.

**Input:** A `.pt` checkpoint from your final Colab training, OR a `SoundScape_CV_Final_Artifacts.zip` containing it. **Output:** `SoundScape_CV_Render_Files.zip` with `models/scene_mobilenetv3.onnx` and `models/scene_mobilenetv3_metadata.json`.

**Runtime:** Google Colab CPU or GPU; no GPU needed for export. If your old Colab session is closed and you did not save its checkpoint/archive, you must recover the file from Drive or rerun training.

In [ ]:
%pip -q install "onnx>=1.16,<2" "onnxruntime>=1.19,<2"

In [ ]:
from pathlib import Path
import io, json, zipfile, shutil
import numpy as np
import torch, onnx, onnxruntime as ort
from torchvision.models import mobilenet_v3_small
from google.colab import files

ROOT = Path('/content/soundscape_cv_bridge')
ROOT.mkdir(parents=True, exist_ok=True)
MODEL_DIR = ROOT / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print('PyTorch:', torch.__version__, 'ONNX:', onnx.__version__)

## 1. Find or upload the *trained* checkpoint

If you just trained in this Colab session, the cell checks `/content/soundscape_cv_final/scene_mobilenetv3.pt` automatically. Otherwise it offers an upload chooser for the `.pt` or the previously downloaded artifacts ZIP. Only use files you created/trust; never load unknown checkpoints.

In [ ]:
checkpoint_path = Path('/content/soundscape_cv_final/scene_mobilenetv3.pt')
if not checkpoint_path.exists():
    print('Upload your scene_mobilenetv3.pt OR SoundScape_CV_Final_Artifacts.zip')
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload exactly one checkpoint or archive')
    filename, blob = next(iter(uploaded.items()))
    if filename.lower().endswith('.pt'):
        checkpoint_path = ROOT / 'scene_mobilenetv3.pt'
        checkpoint_path.write_bytes(blob)
    elif filename.lower().endswith('.zip'):
        with zipfile.ZipFile(io.BytesIO(blob)) as archive:
            choices = [item for item in archive.infolist()
                       if not item.is_dir() and Path(item.filename).name == 'scene_mobilenetv3.pt']
            if len(choices) != 1:
                raise ValueError('Archive must contain exactly one scene_mobilenetv3.pt')
            checkpoint_path = ROOT / 'scene_mobilenetv3.pt'
            with archive.open(choices[0]) as infile, checkpoint_path.open('wb') as outfile:
                shutil.copyfileobj(infile, outfile)
    else:
        raise ValueError('Unsupported file: upload .pt or .zip')
assert checkpoint_path.stat().st_size > 1_000_000, 'File is missing or unexpectedly small'
print('Found checkpoint:', checkpoint_path, f'({checkpoint_path.stat().st_size / 1024**2:.1f} MiB)')

## 2. Export ONNX with calibrated scene metadata

The classifier architecture and class order must match your training checkpoint. This cell uses the actual trained weights, not an ImageNet-only model.

In [ ]:
cp = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
assert isinstance(cp, dict) and isinstance(cp.get('state_dict'), dict), 'Invalid checkpoint structure'
classes = list(cp['classes'])
expected = {'buildings','forest','glacier','mountain','sea','street'}
assert set(classes) == expected and len(classes) == 6, f'Unexpected classes: {classes}'
net = mobilenet_v3_small(weights=None)
net.classifier[3] = torch.nn.Linear(net.classifier[3].in_features, len(classes))
net.load_state_dict(cp['state_dict'], strict=True)
net.eval()
onnx_path = MODEL_DIR / 'scene_mobilenetv3.onnx'
example = torch.rand(1,3,224,224)
with torch.inference_mode():
    original_output = net(example).detach().numpy()
torch.onnx.export(net, example, str(onnx_path),
                  input_names=['pixel_values'], output_names=['logits'],
                  dynamic_axes={'pixel_values':{0:'batch'}, 'logits':{0:'batch'}},
                  opset_version=17, do_constant_folding=True, dynamo=False)
onnx.checker.check_model(str(onnx_path))

mapping = cp.get('vision_to_scene', {
    'buildings':'unknown','forest':'forest','glacier':'mountain',
    'mountain':'mountain','sea':'coast','street':'unknown'
})
metadata = {
    'classes':classes,
    'temperature':float(cp.get('temperature', 1.0)),
    'accept_threshold':float(cp.get('accept_threshold', 0.0)),
    'vision_to_scene':mapping,
    'version':str(cp.get('version','soundscape-cv-export')),
    'training_phase':str(cp.get('training_phase','unknown'))
}
assert metadata['temperature'] > 0
assert 0 <= metadata['accept_threshold'] <= 1
meta_path = MODEL_DIR / 'scene_mobilenetv3_metadata.json'
meta_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print('Exported:', onnx_path.name, meta_path.name)
print('Training phase:', metadata['training_phase'], 'Threshold:',metadata['accept_threshold'])

## 3. Verify PyTorch and ONNX agree

Test the exported model before connecting it to the website. If predicted classes or numerical outputs differ materially, stop rather than shipping a broken classifier.

In [ ]:
session = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
onnx_output = session.run(None, {'pixel_values': example.numpy()})[0]
assert original_output.shape == onnx_output.shape == (1,len(classes))
maximum_error = float(np.max(np.abs(original_output - onnx_output)))
assert np.isfinite(onnx_output).all(), 'Non-finite ONNX output'
assert maximum_error < 0.01, f'ONNX export differs from PyTorch (max error {maximum_error})'
assert int(original_output.argmax()) == int(onnx_output.argmax()), 'Predicted class changed on export'
print('PASSED — ONNX vs PyTorch max absolute logit error:',round(maximum_error,6))
print('Predicted test class:', classes[int(onnx_output.argmax())])

## 4. Download files for your GitHub / Render website

Extract the ZIP on your Mac and copy **the `models` folder** into your local Git clone (`~/Desktop/SoundScape_Deploy`). Git-push from there. Render auto-deploys the update. No need to upload the `.pt` or training dataset to the website.

In [ ]:
result_zip = Path('/content/SoundScape_CV_Render_Files.zip')
with zipfile.ZipFile(result_zip,'w',compression=zipfile.ZIP_DEFLATED) as z:
    z.write(onnx_path,arcname='models/scene_mobilenetv3.onnx')
    z.write(meta_path,arcname='models/scene_mobilenetv3_metadata.json')
print('READY:',result_zip,'size:',round(result_zip.stat().st_size/1024**2,2),'MiB')
files.download(str(result_zip))

## 5. Verify the live site after Git push

Open `https://soundscape-ai-studio.onrender.com/api/status`. Expected: `vision` changes to `onnx_trained_cpu_model_ready`. Then upload an image at the website and click **Analyze this scene**. Select the sound and press **Create audio prompt**.

You should see the six-class model's label/scene suggestion. The model does not infer rain; the user chooses rain intensity manually.

**Note:** The PDF report you previously shared shows completed training; the website requires the actual `.pt` to export real weights. Do not replace it with randomly initialized weights.